In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from pandas_profiling import ProfileReport

C:\Users\umair\AppData\Local\Temp\ipykernel_20524\1882746604.py:6: DeprecationWarning: `import pandas_profiling` is going to be deprecated by April 1st. Please use `import ydata_profiling` instead.
  from pandas_profiling import ProfileReport


In [2]:
df = pd.read_csv("nids.csv")
df
report=ProfileReport(df)
report.to_file("nids_report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 8/8 [00:00<?, ?it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [3]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

In [4]:
k = 3
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
dbscan=DBSCAN(eps=0.5, min_samples=5)
kclusters = kmeans.fit_predict(X_scaled)
dclusters = dbscan.fit_predict(X_scaled)


df["kcluster"] = kclusters
df["dcluster"] = dclusters

In [5]:
kcentroids = kmeans.cluster_centers_
dcentroids = dbscan.components_

kdistances = np.linalg.norm(
    X_scaled - kcentroids[kclusters],
    axis=1
)

ddistances = np.linalg.norm(
    X_scaled - dcentroids[dclusters],
    axis=1
)
df["kanomaly_score"] = kdistances
df["danomaly_score"] = ddistances


In [6]:
threshold = 90
df["kanomaly"] = (df["kanomaly_score"] > threshold).astype(int)
df["danomaly"] = (df["danomaly_score"] > threshold).astype(int)

In [7]:
ksil_score = silhouette_score(X_scaled, kclusters)
dsil_score = silhouette_score(X_scaled, dclusters)
print("Silhouette Score for KMeans:", ksil_score)
print("Silhouette Score for DBSCAN:", dsil_score)

Silhouette Score for KMeans: 0.7052306580324929
Silhouette Score for DBSCAN: 0.44914028378512894


In [8]:
kcluster_anomaly_density = df.groupby("kcluster")["kanomaly"].mean()
dcluster_anomaly_density = df.groupby("dcluster")["danomaly"].mean()
print("KMeans Cluster Anomaly Density:")
print(kcluster_anomaly_density)
print("DBSCAN Cluster Anomaly Density:")    
print(dcluster_anomaly_density)

KMeans Cluster Anomaly Density:
kcluster
0    0.0
1    0.0
2    0.0
Name: kanomaly, dtype: float64
DBSCAN Cluster Anomaly Density:
dcluster
-1    0.0
 0    0.0
 1    0.0
 2    0.0
 3    0.0
Name: danomaly, dtype: float64


In [9]:
print(df[[
    "kcluster",
    "dcluster",
    "kanomaly_score",
    "danomaly_score",
    "kanomaly",
    "danomaly"
]].head())


   kcluster  dcluster  kanomaly_score  danomaly_score  kanomaly  danomaly
0         2         0        0.181323        0.000000         0         0
1         2         0        0.136705        0.278466         0         0
2         0         1        0.242587        1.383984         0         0
3         2         0        0.322914        0.211571         0         0
4         2         0        0.365194        0.395511         0         0
